# Min-K% & Min-K%++ Minimal Research Prototype

**Goal:** Implement minimum viable Min-K% and Min-K%++ membership inference scoring
on the local `mini_gpt_attnres` project, **without modifying any repository source code**.

All implementation is notebook glue code only.

In [ ]:
# =============================================================================
# Cell 1. Environment & Imports
# =============================================================================
import sys
import json
import math
from copy import deepcopy
from pathlib import Path
from collections import defaultdict

import torch
import matplotlib.pyplot as plt
import numpy as np

# --- locate repo root ---
CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = None
for p in CANDIDATES:
    if (p / 'mini_gpt_attnres').exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError('repo root not found (should contain mini_gpt_attnres/)')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from mini_gpt_attnres.evaluate import load_checkpoint
from mini_gpt_attnres.config import DataConfig, ModelConfig
from mini_gpt_attnres.data import SyntheticSequenceDataset

print('REPO_ROOT:', REPO_ROOT)
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('Imports OK')

## Cell 2. Method Definitions (Locked)

### Min-K% (Shi et al., 2023)
**Paper:** *Detecting Pretraining Data from Large Language Models*  
**Repo:** [swj0419/detect-pretrain-code-contamination](https://github.com/swj0419/detect-pretrain-code-contamination)

**Definition:**
Given a sequence $X = (x_1, \ldots, x_N)$ and a target model $M$:
1. Teacher-forced forward: compute $\log p_M(x_t \mid x_{<t})$ for each $t$.
2. Sort the $N$ per-token log-probabilities in ascending order.
3. Select the bottom $k\%$: $k_{\text{len}} = \lfloor N \cdot k / 100 \rfloor$ tokens with the lowest log-prob.
4. Score: $\text{Min-K\%}(X) = \frac{1}{k_{\text{len}}} \sum_{i \in \text{bottom-}k\%} \log p_M(x_i \mid x_{<i})$

Higher (less negative) score $\Rightarrow$ more likely member.

### Min-K%++ (Zhang et al., 2024)
**Paper:** *Min-K%++: Improved Baseline for Detecting Pre-Training Data from Large Language Models*  
**Repo:** [zjysteven/mink-plus-plus](https://github.com/zjysteven/mink-plus-plus)

**Definition:**
Same setup, but each token's log-prob is **z-score normalized** using the model's own
next-token distribution at that position:
1. Compute $\log p_M(v \mid x_{<t})$ and $p_M(v \mid x_{<t})$ for ALL vocab tokens $v$ at position $t$.
2. $\mu_t = \sum_{v \in V} p_M(v \mid x_{<t}) \cdot \log p_M(v \mid x_{<t})$ (probability-weighted mean = negative entropy-like quantity)
3. $\sigma_t^2 = \sum_{v \in V} p_M(v \mid x_{<t}) \cdot [\log p_M(v \mid x_{<t})]^2 - \mu_t^2$ (variance under the model distribution)
4. $z_t = \frac{\log p_M(x_t \mid x_{<t}) - \mu_t}{\sigma_t}$ (z-score normalization)
5. Score: $\text{Min-K\%++}(X) = \frac{1}{k_{\text{len}}} \sum_{i \in \text{bottom-}k\% \text{ of } z} z_i$

**Key:** $\mu_t$ and $\sigma_t$ are computed from the **probability-weighted** expectation
$\mathbb{E}_{V \sim p_M(\cdot|x_{<t})}[\log p_M(V|x_{<t})]$, NOT a uniform average over the vocabulary.
This is the correct implementation per the [official repo](https://github.com/zjysteven/mink-plus-plus).
No reference model, no calibration corpus, no extra samples needed.

### Implementation status
- **Min-K%:** Precise reproduction of the paper formula. Only simplification: we use
  synthetic data instead of natural language, and a tiny model. The scoring math is exact.
- **Min-K%++:** Precise reproduction of the z-score formula. $\mu_t$ and $\sigma_t$ are
  computed over the **full vocabulary** at each position using `probs * log_probs`
  (probability-weighted), matching the official repo. No approximation in the formula.
- **Simplifications vs paper:** (a) tiny model not real LLM, (b) synthetic data not natural
  language benchmark, (c) member/non-member defined by train-seed vs val-seed (smoke test),
  (d) small sample sizes. These are **data/scale simplifications**, not formula approximations.
- This is a **"minimal viable prototype"** -- formula-exact, data-approximate.

In [ ]:
# =============================================================================
# Cell 3. Load Checkpoint & Model
# =============================================================================
# Load both trained TinyStories dual-run checkpoints in one notebook.

MODEL_VARIANTS = ['standard', 'attnres']

CKPT_DIRS = {
    'standard': Path('/scratch/tz6g22/FINAL_report_project/mini_gpt_attnres_runs/tinystories_dual/standard'),
    'attnres': Path('/scratch/tz6g22/FINAL_report_project/mini_gpt_attnres_runs/tinystories_dual/attnres'),
}
CKPT_FILES = {
    'standard': 'ckpt_standard_last.pt',
    'attnres': 'ckpt_attnres_last.pt',
}

unknown = [v for v in MODEL_VARIANTS if v not in CKPT_DIRS]
if unknown:
    raise ValueError(f'Unknown model variant(s): {unknown}. Valid: {list(CKPT_DIRS.keys())}')

device = torch.device('cpu')
models = {}
experiments = {}
checkpoints = {}

for variant in MODEL_VARIANTS:
    ckpt_dir = CKPT_DIRS[variant]
    ckpt_path = ckpt_dir / CKPT_FILES[variant]
    if not ckpt_path.exists():
        raise FileNotFoundError(
            f'Checkpoint not found for variant={variant}: {ckpt_path}\n'
            f'Available config: {json.dumps({k: str(v / CKPT_FILES[k]) for k, v in CKPT_DIRS.items()}, indent=2)}'
        )

    model_i, experiment_i, checkpoint_i = load_checkpoint(ckpt_path, device=device)
    model_i = model_i.to(device).eval()
    models[variant] = model_i
    experiments[variant] = experiment_i
    checkpoints[variant] = checkpoint_i

    mc_i = experiment_i.model
    print(f'[{variant}] checkpoint_path: {ckpt_path}')
    print(f'[{variant}] checkpoint_size: {ckpt_path.stat().st_size / 1024 / 1024:.2f} MB')
    print(f'[{variant}] model_type: {checkpoint_i["model_type"]}, step: {checkpoint_i["step"]}')
    print(f'[{variant}] n_layer={mc_i.n_layer} n_head={mc_i.n_head} n_embd={mc_i.n_embd}')
    print(f'[{variant}] vocab_size={mc_i.vocab_size} block_size={mc_i.block_size}')
    print(f'[{variant}] params={model_i.num_parameters():,}')
    print('-' * 80)

# Backward-compatible handles (default to standard for shared data prep cells).
model = models['standard']
experiment = experiments['standard']
checkpoint = checkpoints['standard']
mc = experiment.model

In [ ]:
# =============================================================================
# Cell 4. Prepare Samples (member vs non-member)
# =============================================================================
# Strategy:
#   member     = training set samples (same seed=1234 as training)
#   non-member = validation set samples (seed=1235, different seed)
#
# Both use the SAME data config as the checkpoint's training run.
# This is a smoke test: synthetic repeated_pattern train-vs-val, NOT a
# rigorous benchmark. The model saw the member sequences during training
# (though only 2 steps), and never saw the non-member sequences.

dc = deepcopy(experiment.data)
train_seed = experiment.train.seed

# SyntheticSequenceDataset only supports random / repeated_pattern / retrieval.
# TinyStories checkpoints carry dataset_type='tinystories', so we fall back to a
# synthetic smoke-test config for the Min-K pipeline check.
supported_synth_types = {'random', 'repeated_pattern', 'retrieval'}
if dc.dataset_type not in supported_synth_types:
    print(
        f"[info] checkpoint data.dataset_type='{dc.dataset_type}' is not supported by SyntheticSequenceDataset; "
        "using repeated_pattern smoke-test samples instead."
    )
    dc = DataConfig(
        dataset_type='repeated_pattern',
        train_size=128,
        val_size=128,
        pattern_length=8,
        retrieval_pairs=4,
    )

N_MEMBER = 32
N_NONMEMBER = 32

# Build member samples (same seed as training -> same sequences)
member_dataset = SyntheticSequenceDataset(
    size=N_MEMBER,
    model_config=mc,
    data_config=dc,
    seed=train_seed,
)

# Build non-member samples (different seed -> different sequences)
nonmember_dataset = SyntheticSequenceDataset(
    size=N_NONMEMBER,
    model_config=mc,
    data_config=dc,
    seed=train_seed + 1,  # val seed used during training
)

# Extract (input, target) pairs
member_inputs = torch.stack([member_dataset[i][0] for i in range(N_MEMBER)])
member_targets = torch.stack([member_dataset[i][1] for i in range(N_MEMBER)])
nonmember_inputs = torch.stack([nonmember_dataset[i][0] for i in range(N_NONMEMBER)])
nonmember_targets = torch.stack([nonmember_dataset[i][1] for i in range(N_NONMEMBER)])

print(f'Dataset type:     {dc.dataset_type}')
print(f'Pattern length:   {dc.pattern_length}')
print(f'Member samples:   {N_MEMBER}, shape: {member_inputs.shape}')
print(f'Non-member samples: {N_NONMEMBER}, shape: {nonmember_inputs.shape}')
print(f'Sequence length (input): {member_inputs.shape[1]} (= block_size = {mc.block_size})')
print(f'All within block_size: {member_inputs.shape[1] <= mc.block_size}')
print(f'\nSample member[0] input[:8]:     {member_inputs[0, :8].tolist()}')
print(f'Sample member[0] target[:8]:    {member_targets[0, :8].tolist()}')
print(f'Sample nonmember[0] input[:8]:  {nonmember_inputs[0, :8].tolist()}')
print(f'Sample nonmember[0] target[:8]: {nonmember_targets[0, :8].tolist()}')

In [ ]:
# =============================================================================
# Cell 5. Teacher-Forced Token Scoring Helper (notebook glue code)
# =============================================================================
# This helper takes input_ids and targets, runs the model, and returns
# per-token log-probabilities. No source code modification needed.
#
# Alignment note:
#   input_ids[t] -> logits[t] predicts target[t] = input_ids[t+1]
#   So logits[:, t, :] is the distribution for the NEXT token after position t.
#   We compute log_softmax(logits[:, t, :]) and gather at target[t].
#   This is the standard teacher-forced setup, matching how the model was trained.

@torch.no_grad()
def compute_per_token_logprobs(input_ids: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Compute per-token log-probability under teacher forcing.
    
    Args:
        input_ids: [B, L] token ids fed to the model
        targets:   [B, L] the ground-truth next tokens (shifted by 1)
    
    Returns:
        token_logprobs: [B, L] log p(target[t] | input_ids[:t+1]) for each t
    """
    outputs = model(input_ids.to(device))
    logits = outputs['logits']  # [B, L, V]
    log_probs = torch.log_softmax(logits, dim=-1)  # [B, L, V]
    # Gather the log-prob of the actual target token at each position
    token_logprobs = torch.gather(
        log_probs, dim=-1, index=targets.to(device).unsqueeze(-1)
    ).squeeze(-1)  # [B, L]
    return token_logprobs.cpu()


@torch.no_grad()
def compute_per_token_logprobs_and_stats(input_ids: torch.Tensor, targets: torch.Tensor):
    """
    Extended version: also returns per-position mu and sigma for Min-K%++.
    
    For Min-K%++, we need at each position t:
      mu_t    = sum_v p(v|x_{<t}) * log p(v|x_{<t})     (prob-weighted mean of log-probs)
      sigma_t = sqrt( sum_v p(v|x_{<t}) * [log p(v|x_{<t})]^2 - mu_t^2 )
    
    Returns:
        token_logprobs: [B, L]  log p(x_t | x_{<t})
        mu:             [B, L]  prob-weighted mean of log-probs over vocab
        sigma:          [B, L]  prob-weighted std of log-probs over vocab
    """
    outputs = model(input_ids.to(device))
    logits = outputs['logits']  # [B, L, V]
    
    log_probs = torch.log_softmax(logits, dim=-1)  # [B, L, V]
    probs = torch.softmax(logits, dim=-1)           # [B, L, V]
    
    # Per-token log-prob of actual target
    token_logprobs = torch.gather(
        log_probs, dim=-1, index=targets.to(device).unsqueeze(-1)
    ).squeeze(-1)  # [B, L]
    
    # Prob-weighted mean: mu_t = E_p[log p] = sum_v p(v) * log p(v)
    mu = (probs * log_probs).sum(dim=-1)  # [B, L]
    
    # Prob-weighted second moment: E_p[(log p)^2] = sum_v p(v) * (log p(v))^2
    second_moment = (probs * log_probs.pow(2)).sum(dim=-1)  # [B, L]
    
    # Variance = E[(log p)^2] - (E[log p])^2, clamped to avoid sqrt(neg)
    variance = (second_moment - mu.pow(2)).clamp(min=1e-10)
    sigma = variance.sqrt()  # [B, L]
    
    return token_logprobs.cpu(), mu.cpu(), sigma.cpu()


# --- Quick sanity check ---
test_lp = compute_per_token_logprobs(member_inputs[:2], member_targets[:2])
print(f'Sanity check: token_logprobs shape = {test_lp.shape}')
print(f'Sample logprobs[0]: {test_lp[0].tolist()}')
print(f'Mean logprob[0]:    {test_lp[0].mean().item():.4f}')
print(f'All finite: {torch.isfinite(test_lp).all().item()}')

In [ ]:
# =============================================================================
# Cell 6. Min-K% Implementation
# =============================================================================
# Exact reproduction of Shi et al. (2023):
#   1. Get per-token logprobs (teacher-forced)
#   2. Sort ascending
#   3. Take bottom k% tokens
#   4. Average their logprobs -> Min-K% score
#
# Higher (less negative) = more likely member.

def min_k_percent_score(token_logprobs: torch.Tensor, k_percent: float) -> torch.Tensor:
    """
    Compute Min-K% score for a batch of sequences.
    
    Args:
        token_logprobs: [B, L] per-token log-probabilities
        k_percent:      float in (0, 100], percentage of lowest-logprob tokens to select
    
    Returns:
        scores: [B] Min-K% score for each sequence
    """
    B, L = token_logprobs.shape
    k_len = max(1, int(L * k_percent / 100.0))
    
    # Sort ascending, take the k_len smallest values
    sorted_logprobs, _ = torch.sort(token_logprobs, dim=-1)  # ascending
    bottom_k = sorted_logprobs[:, :k_len]  # [B, k_len]
    
    # Average
    scores = bottom_k.mean(dim=-1)  # [B]
    return scores


# --- Verify on a single example ---
test_scores = min_k_percent_score(test_lp, k_percent=20.0)
print(f'Min-K%(20%) scores for 2 test samples: {test_scores.tolist()}')

# Show how many tokens are selected at each k
L = member_inputs.shape[1]
for k in [10, 20, 40, 60]:
    k_len = max(1, int(L * k / 100.0))
    print(f'  k={k}%: selecting {k_len}/{L} tokens')

In [ ]:
# =============================================================================
# Cell 7. Min-K%++ Implementation (z-score normalized)
# =============================================================================
# Exact reproduction of Zhang et al. (2024):
#   1. Get per-token logprobs AND per-position mu, sigma
#   2. z_t = (logprob_t - mu_t) / sigma_t
#   3. Sort z-scores ascending, take bottom k%
#   4. Average -> Min-K%++ score
#
# Key difference from Min-K%:
#   Min-K% uses raw log-probs.
#   Min-K%++ normalizes each token's logprob by the model's own expected logprob
#   distribution at that position, making the score robust to position-dependent
#   or context-dependent variation in the model's confidence.
#
# mu_t = E_{V~p}[log p(V|x_{<t})] = sum_v p(v|x_{<t}) * log p(v|x_{<t})
# sigma_t = sqrt(Var_{V~p}[log p(V|x_{<t})])
# These are probability-weighted (not uniform) — matching the official repo.
# No reference model needed. Single forward pass.

def min_k_pp_score(
    token_logprobs: torch.Tensor,
    mu: torch.Tensor,
    sigma: torch.Tensor,
    k_percent: float,
) -> torch.Tensor:
    """
    Compute Min-K%++ score for a batch of sequences.
    
    Args:
        token_logprobs: [B, L] per-token log-probabilities
        mu:             [B, L] prob-weighted mean of log-probs at each position
        sigma:          [B, L] prob-weighted std of log-probs at each position
        k_percent:      float in (0, 100]
    
    Returns:
        scores: [B] Min-K%++ score for each sequence
    """
    B, L = token_logprobs.shape
    k_len = max(1, int(L * k_percent / 100.0))
    
    # Z-score normalization per token
    z = (token_logprobs - mu) / sigma.clamp(min=1e-8)  # [B, L]
    
    # Sort ascending, take bottom k%
    sorted_z, _ = torch.sort(z, dim=-1)
    bottom_k = sorted_z[:, :k_len]  # [B, k_len]
    
    # Average
    scores = bottom_k.mean(dim=-1)  # [B]
    return scores


# --- Verify on a single example ---
test_lp2, test_mu, test_sigma = compute_per_token_logprobs_and_stats(
    member_inputs[:2], member_targets[:2]
)
test_pp_scores = min_k_pp_score(test_lp2, test_mu, test_sigma, k_percent=20.0)
print(f'Min-K%++(20%) scores for 2 test samples: {test_pp_scores.tolist()}')
print(f'\nPer-position stats for sample 0:')
print(f'  logprob: {test_lp2[0].tolist()}')
print(f'  mu:      {test_mu[0].tolist()}')
print(f'  sigma:   {test_sigma[0].tolist()}')
z_example = (test_lp2[0] - test_mu[0]) / test_sigma[0].clamp(min=1e-8)
print(f'  z-score: {z_example.tolist()}')

In [ ]:
# =============================================================================
# Cell 8. Score All Samples
# =============================================================================
# Run Min-K% and Min-K%++ for BOTH models in one pass.
# k values: 10%, 20%, 40%

K_VALUES = [10, 20, 40]
results_by_variant = {}

for variant in MODEL_VARIANTS:
    print(f'\n=== Scoring variant: {variant} ===')
    model = models[variant]

    mem_logprobs, mem_mu, mem_sigma = compute_per_token_logprobs_and_stats(
        member_inputs, member_targets
    )
    nonmem_logprobs, nonmem_mu, nonmem_sigma = compute_per_token_logprobs_and_stats(
        nonmember_inputs, nonmember_targets
    )

    mem_avg_logprob = mem_logprobs.mean(dim=-1)
    nonmem_avg_logprob = nonmem_logprobs.mean(dim=-1)

    variant_results = defaultdict(dict)
    for k in K_VALUES:
        variant_results[f'MinK_{k}']['member'] = min_k_percent_score(mem_logprobs, k).numpy()
        variant_results[f'MinK_{k}']['nonmember'] = min_k_percent_score(nonmem_logprobs, k).numpy()
        variant_results[f'MinKPP_{k}']['member'] = min_k_pp_score(mem_logprobs, mem_mu, mem_sigma, k).numpy()
        variant_results[f'MinKPP_{k}']['nonmember'] = min_k_pp_score(nonmem_logprobs, nonmem_mu, nonmem_sigma, k).numpy()

    variant_results['avg_logprob']['member'] = mem_avg_logprob.numpy()
    variant_results['avg_logprob']['nonmember'] = nonmem_avg_logprob.numpy()

    results_by_variant[variant] = variant_results

    print(f'[{variant}] done: member={len(mem_avg_logprob)}, non-member={len(nonmem_avg_logprob)}')

# Backward compatibility aliases (standard)
results = results_by_variant['standard']

In [ ]:
# =============================================================================
# Cell 9. Aggregate Statistics & Simple Discrimination
# =============================================================================

def compute_auroc_manual(member_scores, nonmember_scores):
    """
    Manual AUC-ROC computation (no sklearn dependency).
    Convention: higher score = more likely member (label=1).
    Uses the Mann-Whitney U statistic formulation.
    """
    m_scores = np.asarray(member_scores)
    nm_scores = np.asarray(nonmember_scores)
    n_m = len(m_scores)
    n_nm = len(nm_scores)
    if n_m == 0 or n_nm == 0:
        return float('nan')
    
    # For each (member, nonmember) pair, count how often member > nonmember
    u_count = 0
    for ms in m_scores:
        u_count += np.sum(ms > nm_scores) + 0.5 * np.sum(ms == nm_scores)
    auc = u_count / (n_m * n_nm)
    return auc


auc_results_by_variant = {}
metric_list = ['avg_logprob'] + [f'MinK_{k}' for k in K_VALUES] + [f'MinKPP_{k}' for k in K_VALUES]

for variant in MODEL_VARIANTS:
    print(f'\n=== Aggregate Statistics: {variant} ===')
    print(f'{"metric":>15s}  {"mem_mean":>9s}  {"mem_std":>9s}  {"nonmem_mean":>11s}  {"nonmem_std":>10s}  {"AUC-ROC":>8s}')
    print('-' * 75)

    results_i = results_by_variant[variant]
    auc_results = {}
    for metric_name in metric_list:
        m = results_i[metric_name]['member']
        nm = results_i[metric_name]['nonmember']
        auc = compute_auroc_manual(m, nm)
        auc_results[metric_name] = auc
        print(f'{metric_name:>15s}  {np.mean(m):9.4f}  {np.std(m):9.4f}  {np.mean(nm):11.4f}  {np.std(nm):10.4f}  {auc:8.4f}')

    auc_results_by_variant[variant] = auc_results

print('\n--- Interpretation ---')
print('AUC-ROC = 0.5 means random guessing (no discrimination).')
print('AUC-ROC > 0.5 means member scores tend to be higher than non-member.')
print('AUC-ROC = 1.0 means perfect discrimination.')
print('\nNOTE: This is a SMOKE TEST on tiny checkpoints.')

In [ ]:
# =============================================================================
# Cell 10. Visualization (ONE combined figure for standard + attnres)
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Min-K% / Min-K%++ Combined Results (standard + attnres)', fontsize=14)

# Panel A: AUC vs k for Min-K%
ax = axes[0, 0]
for variant in MODEL_VARIANTS:
    auc_curve = [auc_results_by_variant[variant][f'MinK_{k}'] for k in K_VALUES]
    ax.plot(K_VALUES, auc_curve, marker='o', linewidth=2, label=variant)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_title('AUC-ROC vs k (Min-K%)')
ax.set_xlabel('k (%)')
ax.set_ylabel('AUC-ROC')
ax.set_ylim(0.0, 1.0)
ax.legend()
ax.grid(alpha=0.3)

# Panel B: AUC vs k for Min-K%++
ax = axes[0, 1]
for variant in MODEL_VARIANTS:
    auc_curve = [auc_results_by_variant[variant][f'MinKPP_{k}'] for k in K_VALUES]
    ax.plot(K_VALUES, auc_curve, marker='o', linewidth=2, label=variant)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_title('AUC-ROC vs k (Min-K%++)')
ax.set_xlabel('k (%)')
ax.set_ylabel('AUC-ROC')
ax.set_ylim(0.0, 1.0)
ax.legend()
ax.grid(alpha=0.3)

# Panel C: Score distributions (Min-K%++, k=20) for both models
ax = axes[1, 0]
bins = 25
for variant in MODEL_VARIANTS:
    m = results_by_variant[variant]['MinKPP_20']['member']
    nm = results_by_variant[variant]['MinKPP_20']['nonmember']
    ax.hist(m, bins=bins, histtype='step', linewidth=2, label=f'{variant} member')
    ax.hist(nm, bins=bins, histtype='step', linewidth=2, linestyle='--', label=f'{variant} non-member')
ax.set_title('Min-K%++ score distribution (k=20)')
ax.set_xlabel('score')
ax.set_ylabel('count')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel D: AUC summary bars
ax = axes[1, 1]
summary_metrics = ['avg_logprob', 'MinK_20', 'MinKPP_20']
x = np.arange(len(summary_metrics))
w = 0.35
std_vals = [auc_results_by_variant['standard'][m] for m in summary_metrics]
att_vals = [auc_results_by_variant['attnres'][m] for m in summary_metrics]
ax.bar(x - w / 2, std_vals, width=w, label='standard')
ax.bar(x + w / 2, att_vals, width=w, label='attnres')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(summary_metrics)
ax.set_ylim(0.0, 1.0)
ax.set_title('AUC summary (baseline / Min-K / Min-K++)')
ax.set_ylabel('AUC-ROC')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.show()
print('Combined visualization complete.')

## Cell 11. Results Interpretation

### What this notebook implements

1. **Min-K% (Shi et al., 2023)** -- exact formula reproduction:
   - Teacher-forced per-token logprob via `log_softmax + gather`
   - Bottom k% selection and averaging
   - Tested at k = 10%, 20%, 40%

2. **Min-K%++ (Zhang et al., 2024)** -- exact formula reproduction:
   - Per-position probability-weighted mean ($\mu_t$) and std ($\sigma_t$) over full vocab
   - Z-score normalization: $(\log p - \mu) / \sigma$
   - Bottom k% of z-scores, then average
   - No reference model, no calibration corpus -- single forward pass

3. **Baseline**: average log-probability (full sequence, no selection)

### What is precise vs simplified

| Aspect | Status |
|--------|--------|
| Min-K% formula | **Exact** -- matches paper and official repo |
| Min-K%++ formula | **Exact** -- probability-weighted mu/sigma, matching official repo |
| Per-token logprob computation | **Exact** -- standard teacher-forced log_softmax + gather |
| Model | **Simplified** -- tiny (117K params), 2 training steps, NOT a real LLM |
| Data | **Simplified** -- synthetic `repeated_pattern`, NOT natural language |
| Member/non-member definition | **Smoke test only** -- train-seed vs val-seed on synthetic data |
| AUC evaluation | **Manual implementation** (Mann-Whitney U), correct but not using sklearn |

### Expected behavior with this checkpoint

The bootstrap checkpoint was trained for **only 2 steps** on a tiny model. This means:
- The model has barely learned anything beyond random initialization
- Member vs non-member discrimination is expected to be **near random** (AUC ~ 0.5)
- This is **by design** for a smoke test -- the goal is to verify the scoring pipeline works
- With a properly trained checkpoint (e.g., the TinyStories checkpoints at `/scratch/`),
  we would expect meaningful discrimination

### Gap from paper-level results

To approach paper-quality results, you would need:
1. A model trained to convergence (not 2 steps)
2. Natural language data (e.g., TinyStories, WikiMIA, etc.)
3. Proper member/non-member splits (e.g., text in training set vs held-out text)
4. Larger sample sizes (hundreds to thousands, not 32)
5. Multiple model sizes for scaling analysis

### What is notebook glue code vs what needs library support

| Component | Where |
|-----------|-------|
| Model loading | Library (`load_checkpoint`) |
| Forward pass | Library (`model(idx)`) |
| Per-token logprob extraction | **Notebook glue** (`log_softmax + gather`) |
| Min-K% scoring | **Notebook glue** (sort + select + mean) |
| Min-K%++ mu/sigma | **Notebook glue** (`probs * log_probs` weighted stats) |
| AUC computation | **Notebook glue** (manual Mann-Whitney U) |
| Data construction | Library (`SyntheticSequenceDataset`) |

If you want to productionize:
- The per-token logprob helper could become a library function
- The Min-K%/Min-K%++ scoring could become utility functions in a new `mia.py` module
- But for research prototyping, notebook glue code is sufficient and correct

### Zero source code modifications

This notebook makes **zero changes** to any file in `mini_gpt_attnres/`. All scoring
logic is implemented as notebook-local functions using the existing public API:
- `load_checkpoint()` from `evaluate.py`
- `model(input_ids)` returning `{"logits": tensor}`
- `SyntheticSequenceDataset` from `data.py`